# 02a — Preprocessing demo and validation (MVTec AD)

This notebook validates the repository's **existing** preprocessing code on the real MVTec AD dataset before PCA. It does not modify the raw dataset or fit any model.

The all-category check is sampled by default for speed. Set `FULL_SCAN = True` for an exhaustive scan.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, copy
from pathlib import Path

REPO_URL = 'https://github.com/pranitas-cloud/CS-663.git'
REPO = Path('/content/CS-663')
if not REPO.exists():
    result = subprocess.run(['git', 'clone', REPO_URL, str(REPO)], capture_output=True, text=True)
else:
    result = subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', 'main'], capture_output=True, text=True)
print(result.stdout + result.stderr)
if result.returncode != 0:
    raise RuntimeError('Could not clone/update CS-663; inspect the Git output above.')

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('Repository:', REPO)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset

from src.data.dataset import MVTecDataset, available_categories
from src.data.pipelines import build_preprocessor, build_dataset, load_config, stack_images
from src.data.preprocessing import image_to_tensor, resize_image, normalize_imagenet, denormalize_imagenet, mask_to_tensor

pd.set_option('display.max_columns', 50)

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection')
assert DATA_ROOT.is_dir(), f'Dataset root not found: {DATA_ROOT}'

cfg = copy.deepcopy(load_config('configs/default.yaml'))
cfg['data']['root'] = str(DATA_ROOT)  # in-memory override only; YAML is unchanged
categories = available_categories(DATA_ROOT)
print(f'{len(categories)} categories:', categories)
assert len(categories) == 15, 'Expected 15 MVTec categories.'

METHODS = ('pca', 'autoencoder', 'patchcore')
preprocessors = {m: build_preprocessor(m, cfg) for m in METHODS}
for name, pre in preprocessors.items():
    print(name, pre.describe())

## 2. Representative RGB and grayscale examples

We use a defective test image from `bottle` (RGB-stored) and `grid` (grayscale-stored). These are visualization examples only; they are not used for training.

In [ ]:
REPRESENTATIVE = {'bottle (RGB)': 'bottle', 'grid (grayscale)': 'grid'}

def first_defective_item(category):
    ds = MVTecDataset(DATA_ROOT, category, split='test')
    for i, sample in enumerate(ds.samples):
        if sample[2] != 'good':
            return ds[i]
    raise RuntimeError(f'No defective test image found for {category}')

raw_items = {}
for label, category in REPRESENTATIVE.items():
    item = first_defective_item(category)
    raw_items[category] = item
    image, mask = item['image'], item['mask']
    print(f'{category}: defect={item["defect_type"]}, image={image.shape}/{image.dtype}, mask={mask.shape}/{mask.dtype}, mask values={np.unique(mask).tolist()}')

assert raw_items['bottle']['image'].ndim == 3
assert raw_items['grid']['image'].ndim == 2

## 3. Preprocessing steps and visual comparison

In [ ]:
IMAGE_SIZE = cfg['data']['image_size']

def tensor_stats(t):
    return {'shape': tuple(t.shape), 'dtype': str(t.dtype), 'min': float(t.min()),
            'max': float(t.max()), 'mean': float(t.mean()), 'std': float(t.std()),
            'finite': bool(torch.isfinite(t).all())}

step_rows = []
for category, item in raw_items.items():
    image = item['image']
    steps = {
        'RGB tensor': image_to_tensor(image, grayscale=False),
        'grayscale tensor': image_to_tensor(image, grayscale=True),
    }
    steps['resized RGB'] = resize_image(steps['RGB tensor'], IMAGE_SIZE)
    steps['resized grayscale'] = resize_image(steps['grayscale tensor'], IMAGE_SIZE)
    steps['ImageNet normalized RGB'] = normalize_imagenet(steps['resized RGB'])
    steps['processed mask'] = mask_to_tensor(item['mask'], IMAGE_SIZE, mode=cfg['data'].get('mask_resize', 'nearest'))
    for name, tensor in steps.items():
        step_rows.append({'category': category, 'step': name, **tensor_stats(tensor)})
display(pd.DataFrame(step_rows))

In [ ]:
def display_tensor(t):
    a = t.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    return a[..., 0] if a.shape[-1] == 1 else a

fig, axes = plt.subplots(len(raw_items), 4, figsize=(14, 7))
for row, (category, item) in enumerate(raw_items.items()):
    raw = item['image']
    x_pca, _ = preprocessors['pca'](raw, item['mask'])
    x_ae, _ = preprocessors['autoencoder'](raw, item['mask'])
    x_pc, _ = preprocessors['patchcore'](raw, item['mask'])
    axes[row, 0].imshow(raw, cmap='gray' if raw.ndim == 2 else None)
    axes[row, 0].set_title(f'{category}: original')
    axes[row, 1].imshow(display_tensor(x_pca), cmap='gray')
    axes[row, 1].set_title('PCA: grayscale [0,1]')
    axes[row, 2].imshow(display_tensor(x_ae))
    axes[row, 2].set_title('Autoencoder: RGB [0,1]')
    axes[row, 3].imshow(display_tensor(denormalize_imagenet(x_pc)))
    axes[row, 3].set_title('PatchCore: denormalized for display')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Original and resized masks, with overlays to inspect alignment.
fig, axes = plt.subplots(len(raw_items), 4, figsize=(14, 7))
for row, (category, item) in enumerate(raw_items.items()):
    raw, raw_mask = item['image'], item['mask']
    _, processed_mask = preprocessors['pca'](raw, raw_mask)
    m = processed_mask[0].numpy()
    axes[row, 0].imshow(raw_mask, cmap='gray'); axes[row, 0].set_title('Original mask')
    axes[row, 1].imshow(m, cmap='gray', vmin=0, vmax=1); axes[row, 1].set_title('Resized binary mask')
    axes[row, 2].imshow(raw, cmap='gray' if raw.ndim == 2 else None)
    axes[row, 2].imshow(np.ma.masked_where(raw_mask == 0, raw_mask), cmap='autumn', alpha=.5); axes[row, 2].set_title('Original overlay')
    axes[row, 3].imshow(display_tensor(preprocessors['pca'](raw, None)[0]), cmap='gray')
    axes[row, 3].imshow(np.ma.masked_where(m == 0, m), cmap='autumn', alpha=.5); axes[row, 3].set_title('Processed overlay')
    for ax in axes[row]: ax.axis('off')
    print(category, 'defect area:', f'{(raw_mask > 0).mean():.4%} -> {m.mean():.4%}')
plt.tight_layout(); plt.show()

## 4. Full-pipeline tensor statistics

In [ ]:
rows = []
for category, item in raw_items.items():
    for method, pre in preprocessors.items():
        image_tensor, mask_tensor = pre(item['image'], item['mask'])
        rows.append({'category': category, 'method': method, 'tensor': 'image', **tensor_stats(image_tensor)})
        rows.append({'category': category, 'method': method, 'tensor': 'mask', **tensor_stats(mask_tensor)})
display(pd.DataFrame(rows))

gray_ae, _ = preprocessors['autoencoder'](raw_items['grid']['image'], None)
print('Grayscale source replicated into identical AE channels:',
      torch.equal(gray_ae[0], gray_ae[1]) and torch.equal(gray_ae[1], gray_ae[2]))

normal = MVTecDataset(DATA_ROOT, 'bottle', split='train')[0]
_, normal_mask = preprocessors['pca'](normal['image'], normal['mask'])
print('Normal sample has None raw mask:', normal['mask'] is None,
      '| processed mask sum:', normal_mask.sum().item())

## 5. Check all 15 categories

By default, check 2 train images, 1 validation image, and 1 test image per defect type (including `good`). Set `FULL_SCAN = True` to check every image. All three method-specific preprocessors are tested for each sampled image.

In [ ]:
FULL_SCAN = False
N_TRAIN, N_VAL, PER_DEFECT = 2, 1, 1

IM_MEAN = (0.485, 0.456, 0.406)
IM_STD = (0.229, 0.224, 0.225)
PC_LO = min((0 - m) / s for m, s in zip(IM_MEAN, IM_STD)) - 1e-4
PC_HI = max((1 - m) / s for m, s in zip(IM_MEAN, IM_STD)) + 1e-4

def pick_indices(ds, split):
    if FULL_SCAN: return list(range(len(ds)))
    if split == 'train': return list(range(min(N_TRAIN, len(ds))))
    if split == 'val': return list(range(min(N_VAL, len(ds))))
    seen, chosen = {}, []
    for i, sample in enumerate(ds.samples):
        defect = sample[2]
        if seen.get(defect, 0) < PER_DEFECT:
            chosen.append(i)
            seen[defect] = seen.get(defect, 0) + 1
    return chosen

def check_item(item):
    problems, warnings = [], []
    raw, raw_mask, label = item['image'], item['mask'], item['label']
    raw_copy = raw.copy()
    mask_copy = None if raw_mask is None else raw_mask.copy()

    if raw_mask is not None:
        if raw_mask.dtype != np.uint8: problems.append(f'raw mask dtype is {raw_mask.dtype}, expected uint8')
        if not set(np.unique(raw_mask).tolist()) <= {0, 255}: problems.append(f'unexpected raw mask values: {np.unique(raw_mask).tolist()}')
        if raw_mask.shape != raw.shape[:2]: problems.append(f'mask shape {raw_mask.shape} != image shape {raw.shape[:2]}')
        if label == 1 and not (raw_mask > 0).any(): problems.append('anomalous raw mask is empty')

    outputs = {}
    for method, pre in preprocessors.items():
        try:
            x, y = pre(raw, raw_mask)
            outputs[method] = (x, y)
        except Exception as exc:
            problems.append(f'{method} raised {type(exc).__name__}: {exc}')
            continue

        size = pre.image_size
        if tuple(x.shape) != (pre.channels, size, size): problems.append(f'{method} image shape {tuple(x.shape)}')
        if tuple(y.shape) != (1, size, size): problems.append(f'{method} mask shape {tuple(y.shape)}')
        if x.dtype != torch.float32 or y.dtype != torch.float32: problems.append(f'{method} dtype is {x.dtype}/{y.dtype}')
        if not torch.isfinite(x).all(): problems.append(f'{method} image contains NaN/Inf')
        if not torch.isfinite(y).all(): problems.append(f'{method} mask contains NaN/Inf')
        if x.numel() and torch.isfinite(x).all():
            lo, hi = (PC_LO, PC_HI) if method == 'patchcore' else (0., 1.)
            if x.min().item() < lo or x.max().item() > hi: problems.append(f'{method} image range [{x.min().item():.3f}, {x.max().item():.3f}] outside [{lo:.2f}, {hi:.2f}]')
        if not set(y.unique().tolist()) <= {0., 1.}: problems.append(f'{method} mask is not binary')
        if label == 0 and y.sum().item() != 0: problems.append(f'{method} normal image has non-empty mask')
        if label == 1 and y.sum().item() == 0 and method == 'pca': warnings.append('defect vanished after nearest-neighbour resizing')

    if 'autoencoder' in outputs and raw.ndim == 2:
        a = outputs['autoencoder'][0]
        if not (torch.equal(a[0], a[1]) and torch.equal(a[1], a[2])): problems.append('grayscale source was not replicated identically')
    masks = [pair[1] for pair in outputs.values()]
    if len(masks) > 1 and any(not torch.equal(masks[0], other) for other in masks[1:]):
        problems.append('processed masks differ between methods')
    if not np.array_equal(raw, raw_copy): problems.append('raw image array was modified')
    if raw_mask is not None and not np.array_equal(raw_mask, mask_copy): problems.append('raw mask array was modified')

    pca_mask = outputs.get('pca', (None, None))[1]
    info = {'source_mode': 'L' if raw.ndim == 2 else 'RGB',
            'source_size': raw.shape[0],
            'raw_mask_fraction': None if raw_mask is None else float((raw_mask > 0).mean()),
            'processed_mask_fraction': None if pca_mask is None else float(pca_mask.mean())}
    return problems, warnings, info

In [ ]:
records, split_rows = [], []
for category in categories:
    datasets = {split: MVTecDataset(DATA_ROOT, category, split=split,
                  val_fraction=cfg['data']['val_fraction'], seed=cfg['seed'])
                for split in ('train', 'val', 'test')}

    train_paths = {str(s[0]) for s in datasets['train'].samples}
    val_paths = {str(s[0]) for s in datasets['val'].samples}
    test_paths = {str(s[0]) for s in datasets['test'].samples}
    split_rows.append({
        'category': category, 'train': len(train_paths), 'val': len(val_paths), 'test': len(test_paths),
        'train_val_overlap': len(train_paths & val_paths),
        'test_overlap': len(test_paths & (train_paths | val_paths)),
        'non_good_train_val': sum(s[2] != 'good' for split in ('train', 'val') for s in datasets[split].samples)
    })

    for split, ds in datasets.items():
        for index in pick_indices(ds, split):
            item = ds[index]
            problems, warnings, info = check_item(item)
            records.append({'category': category, 'split': split, 'defect': item['defect_type'],
                            'file': Path(item['image_path']).name, **info,
                            'ok': not problems, 'problems': '; '.join(problems),
                            'warnings': '; '.join(warnings)})
    print(category, 'checked:', sum(r['category'] == category for r in records))

results = pd.DataFrame(records)
split_df = pd.DataFrame(split_rows)
print('Samples checked:', len(results), '| failed:', int((~results.ok).sum()),
      '| warnings:', int((results.warnings != '').sum()))

In [ ]:
failed = results[~results.ok]
warned = results[results.warnings != '']
display(failed[['category', 'split', 'defect', 'file', 'problems']] if len(failed) else 'No failed samples')
display(warned[['category', 'split', 'defect', 'file', 'warnings']] if len(warned) else 'No warnings')
display(results.groupby('category').agg(checked=('ok', 'size'), passed=('ok', 'sum'),
        source_modes=('source_mode', lambda s: ', '.join(sorted(set(s))))))
display(split_df)

## 6. PCA input / patch extraction sanity check

This uses only the normal training split. It checks the PCA preprocessing output, flatten/unflatten consistency, and patch extraction/folding. It does **not** fit PCA.

In [ ]:
pca_pre = preprocessors['pca']
pca_rows = []
for category in REPRESENTATIVE.values():
    train_ds = build_dataset(cfg, 'pca', 'train', category)
    subset = Subset(train_ds, range(min(16, len(train_ds))))
    X = stack_images(subset)
    flat = pca_pre.flatten(X)
    patches = pca_pre.patches(X)
    folded = pca_pre.fold(patches)
    pca_rows.append({
        'category': category, 'train_images': len(subset),
        'image_batch_shape': tuple(X.shape), 'flattened_shape': tuple(flat.shape),
        'patches_shape': tuple(patches.shape),
        'unflatten_exact': torch.equal(pca_pre.unflatten(flat), X),
        'patch_roundtrip_max_error': float((folded - X).abs().max()),
        'finite': bool(torch.isfinite(flat).all() and torch.isfinite(patches).all()),
        'train_is_normal_only': all(s[2] == 'good' for s in train_ds.samples)
    })
pca_check = pd.DataFrame(pca_rows)
display(pca_check)
print('Patch grid:', pca_pre.describe()['patch_grid'])

## 7. Final validation verdict

In [ ]:
split_ok = bool((split_df[['train_val_overlap', 'test_overlap', 'non_good_train_val']].to_numpy() == 0).all())
nan_inf = results.problems.str.contains('NaN/Inf', regex=False).any()
mask_problem = results.problems.str.contains('mask', case=False, regex=False).any()
modified = results.problems.str.contains('modified', case=False, regex=False).any()
pca_ok = bool(pca_check.unflatten_exact.all() and pca_check.finite.all()
              and pca_check.train_is_normal_only.all()
              and (pca_check.patch_roundtrip_max_error < 1e-5).all())

checks = {
    'All 15 categories represented': results.category.nunique() == 15,
    'No preprocessing failures in checked samples': bool(results.ok.all()),
    'No NaN/Inf': not nan_inf,
    'No mask-related failures': not mask_problem,
    'Raw arrays unchanged': not modified,
    'Train/validation/test split hygiene': split_ok,
    'PCA matrix and patch operations pass': pca_ok,
}
for label, passed in checks.items():
    print(('PASS  ' if passed else 'FAIL  ') + label)

print(f'\nChecked {len(results)} samples; failures={int((~results.ok).sum())}; warnings={int((results.warnings != "").sum())}')
if int((results.warnings != '').sum()):
    print('Warning: some tiny defects may disappear with nearest-neighbour mask downsampling.')
    print("If this matters for pixel-level evaluation, compare against mask_resize='max' separately.")

if all(checks.values()):
    print('\nVERDICT: READY to start PCA (based on the checks above).')
else:
    print('\nVERDICT: NOT READY. Inspect failures before starting PCA.')
if not FULL_SCAN:
    print('Note: this is a deterministic sample, not a full-dataset scan. Set FULL_SCAN=True for exhaustive validation.')